# JC — congruence lane verification

Charter `docs/CONGRUENCE-CHARTER.md` (commit `49311bad`).  Judges run 1 at `e5d6a1b5`.
Target commit: **`87a206ec`** on branch `claude/congruence-spectrum`.

**Runtime: CPU / high-RAM. NEVER GPU** — Lean cannot use a GPU and it burns compute
units far faster.  Cell 1 refuses to continue if one is attached.

**JC predicts:** zero errors, zero `sorry`, every declaration on
`[propext, Classical.choice, Quot.sound]` or a subset, and the merged-core job count
**+1 exactly** over this branch's own baseline measured *in this same tree*.

No GitHub token here and no push from here.  Artifacts return to the desktop and are
hash-verified before anything is committed.  Disconnect the runtime when done —
units are spent per connected hour even when nothing is running.

In [ ]:
# CELL 1 — refuse a GPU runtime, then clone at the exact SHA.
import subprocess, sys
if subprocess.run('nvidia-smi -L', shell=True, capture_output=True).returncode == 0:
    sys.exit('GPU runtime detected. Switch to CPU / high-RAM and re-run: Lean '
             'cannot use a GPU and it burns compute units much faster.')
print('CPU runtime confirmed.')
!git clone --branch claude/congruence-spectrum --single-branch \
    https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git repo
!cd repo && git rev-parse HEAD && cat lean-toolchain

In [ ]:
# CELL 2 — elan + the pinned toolchain.
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh \
    | sh -s -- -y --default-toolchain none
import os
os.environ['PATH'] = os.path.expanduser('~/.elan/bin') + ':' + os.environ['PATH']
!cd repo && elan toolchain install $(cat lean-toolchain) && lake --version

In [ ]:
# CELL 3 — Mathlib oleans from cache.  This does NOT cover the YangMills tree;
# that is expected and is why cell 6 is the long one.
!cd repo && lake exe cache get 2>&1 | tail -5

In [ ]:
# CELL 4 — preflight.  Fails HERE on any SHA / toolchain / pin mismatch,
# before a second of compute is spent on a stale cache.
!cd repo && python3 scripts/colab_congruence_verify.py . preflight

In [ ]:
# CELL 5 — JC parts 1 and 2: elaborate the module, then the axiom oracle.
# Sentinels carry the real decimal exit code, written atomically; logs are named
# for the mode, never with numeric suffixes.
!cd repo && python3 scripts/colab_congruence_verify.py . elaborate
!cd repo && python3 scripts/colab_congruence_verify.py . oracle

In [ ]:
# CELL 6 — JC part 3: the job count.  BASELINE FIRST, in this same tree --
# a delta against a number measured in somebody else's tree is not a measurement.
# Long; Pro+ keeps this running with the tab closed.
!cd repo && git stash list && git checkout HEAD~1 -- YangMillsCore.lean 2>/dev/null; true
!cd repo && lake build YangMillsCore 2>&1 | tail -2   # BASELINE (module not imported)
!cd repo && git checkout HEAD -- YangMillsCore.lean
!cd repo && lake build YangMillsCore 2>&1 | tail -2   # WITH the module

In [ ]:
# CELL 7 — collect artifacts for hash-verified return to the desktop, then
# DISCONNECT the runtime (Runtime > Disconnect and delete runtime).
!cd repo && sha256sum YangMills/OS/CongruenceSpectrum.lean *.log *.sentinel 2>/dev/null
from google.colab import files
import shutil
shutil.make_archive('/content/jc_artifacts', 'zip', 'repo', '.')
files.download('/content/jc_artifacts.zip')